# Decomposing wealth and residential inequalities in facility delivery in Tanzania

Reproducible analysis code for a **Blinder–Oaxaca-type decomposition** of rural–urban and
richest–poorest inequalities in health facility delivery among women in Tanzania, using the
**2022 TDHS-MIS** women's individual recode (`TZIR81FL.DTA`).

**Author:** Clifford Silver Tarimo, Dar es Salaam Institute of Technology

**Data:** Not included. Freely available from the DHS Program (https://dhsprogram.com/data/)
after registration; download the Tanzania 2022 Individual Recode (Stata) and place `TZIR81FL.DTA`
beside this notebook, or edit `DATA_DIR`.

This notebook reproduces: weighted prevalences with 95% confidence intervals (Table 1), the two
decompositions with confidence intervals (Table 2), linear-probability-model diagnostics, a
missing-data assessment, the formal wealth-by-residence interaction test, and the sensitivity
analyses (ANC 8+ threshold, and excluding ANC). All confidence intervals use a cluster bootstrap
that resamples primary sampling units within strata; a fixed seed ensures reproducibility.

## 1. Environment

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
from scipy import stats
print("Python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)
print("numpy     :", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("scipy     :", __import__("scipy").__version__)

SEED = 2024
N_BOOT = 1000

## 2. Load data

In [ ]:
from pathlib import Path
DATA_DIR = Path(".")
IR_FILE  = DATA_DIR / "TZIR81FL.DTA"
assert IR_FILE.exists(), f"TZIR81FL.DTA not found. Place it in {DATA_DIR.resolve()}"
cols = ["caseid","v005","v021","v022","v023","v024","v025","v106","v190",
        "v467d","m14_1","m15_1","b19_01"]
df = pd.read_stata(IR_FILE, columns=cols, convert_categoricals=True)
df["weight"] = df["v005"] / 1_000_000
print("Individual recode rows:", len(df))

## 3. Outcome and analytic sample
Facility delivery for each woman's most recent live birth (`m15_1`).

In [ ]:
analysis = df[df["m15_1"].notna()].copy()
home = ["respondent's home","other home","tba premises","on the way to the hospital","other"]
analysis["facility_delivery"] = np.where(analysis["m15_1"].isin(home), 0, 1)
print("Analytic sample:", len(analysis))

## 4. Covariates
Maternal education is collapsed to **no education / primary / secondary or higher**; the small
'higher' category (1.1%) is merged with 'secondary' for stable estimation.

In [ ]:
analysis["residence"] = analysis["v025"].astype(str)
analysis["wealth"]    = analysis["v190"].astype(str)
analysis["region"]    = analysis["v024"].astype(str)

edu = analysis["v106"].astype(str).replace({"secondary":"secondary or higher",
                                             "higher":"secondary or higher"})
analysis["education"] = pd.Categorical(edu, ["no education","primary","secondary or higher"], ordered=True)
analysis["wealth"]    = pd.Categorical(analysis["wealth"],
                        ["poorest","poorer","middle","richer","richest"], ordered=True)

anc = pd.to_numeric(analysis["m14_1"], errors="coerce")
anc[analysis["m14_1"].astype(str) == "no antenatal visits"] = 0
analysis["anc_visits"] = anc
analysis["anc_4plus"] = np.where(anc.isna(), np.nan, np.where(anc >= 4, 1, 0))
analysis["anc_8plus"] = np.where(anc.isna(), np.nan, np.where(anc >= 8, 1, 0))
analysis["distance_big_problem"] = np.where(analysis["v467d"].astype(str) == "big problem", 1, 0)

## 5. Missing-data assessment

In [ ]:
n = len(analysis)
print(f"Analytic sample: {n}")
for lab, col in [("Residence","v025"),("Region","v024"),("Wealth","v190"),
                 ("Education","v106"),("Distance","v467d")]:
    m = analysis[col].isna().sum(); print(f"  {lab:10s} missing: {m} ({m/n*100:.2f}%)")
m = int(analysis["anc_4plus"].isna().sum())
print(f"  {'ANC 4+':10s} missing: {m} ({m/n*100:.2f}%)  [\"don't know\" responses]")
decomp = analysis.dropna(subset=["facility_delivery","residence","wealth","education",
                                 "anc_4plus","distance_big_problem","weight"])
print("Complete-case decomposition sample:", len(decomp))

## 6. Cluster bootstrap engine and estimators
One bootstrap pass resamples primary sampling units (`v021`) within strata (`v022`) and recomputes
every statistic, giving design-consistent 95% percentile confidence intervals.

In [ ]:
def wmean(y, w): return float(np.sum(y*w)/np.sum(w))

RU_PREDS = ["wealth","education","anc_4plus","distance_big_problem"]
RP_PREDS = ["residence","education","anc_4plus","distance_big_problem"]
DOM_RU = {"wealth":"Wealth","education":"Education","anc":"ANC 4+","distance":"Distance"}
DOM_RP = {"residence":"Urban residence","education":"Education","anc":"ANC 4+","distance":"Distance"}

def decompose(d, group_col, hi_value, predictors):
    d = d.dropna(subset=predictors+["facility_delivery","weight"]).copy()
    X = pd.get_dummies(d[predictors], drop_first=True).astype(float)
    y = d["facility_delivery"].astype(float).values; w = d["weight"].astype(float).values
    Xi = X.copy(); Xi.insert(0,"intercept",1.0); sw = np.sqrt(w)
    beta = np.linalg.lstsq(Xi.values*sw[:,None], y*sw, rcond=None)[0][1:]
    hi = (d[group_col]==hi_value).values
    md_ = np.average(X[hi],axis=0,weights=w[hi]) - np.average(X[~hi],axis=0,weights=w[~hi])
    contrib = dict(zip(X.columns, md_*beta))
    gap = wmean(y[hi],w[hi]) - wmean(y[~hi],w[~hi])
    return contrib, gap

def _domains(contrib, mapping):
    o = {}
    for col,v in contrib.items():
        for k,dm in mapping.items():
            if col.startswith(k): o[dm] = o.get(dm,0)+v
    return o

def all_stats(a, anc="anc_4plus"):
    out = {}
    out["overall"] = wmean(a["facility_delivery"].values, a["weight"].values)*100
    for lab,m in [("urban",a["residence"]=="urban"),("rural",a["residence"]=="rural")]:
        s=a[m.values]; out[lab]=wmean(s["facility_delivery"].values,s["weight"].values)*100
    for q in ["poorest","poorer","middle","richer","richest"]:
        s=a[a["wealth"]==q]; out["W_"+q]=wmean(s["facility_delivery"].values,s["weight"].values)*100
    for lab in ["no education","primary","secondary or higher"]:
        s=a[a["education"]==lab]; out["E_"+lab]=wmean(s["facility_delivery"].values,s["weight"].values)*100
    da=a.dropna(subset=[anc]); 
    out["anc_yes"]=wmean(da[da[anc]==1]["facility_delivery"].values,da[da[anc]==1]["weight"].values)*100
    out["anc_no"] =wmean(da[da[anc]==0]["facility_delivery"].values,da[da[anc]==0]["weight"].values)*100
    out["dist_yes"]=wmean(a[a["distance_big_problem"]==1]["facility_delivery"].values,a[a["distance_big_problem"]==1]["weight"].values)*100
    out["dist_no"] =wmean(a[a["distance_big_problem"]==0]["facility_delivery"].values,a[a["distance_big_problem"]==0]["weight"].values)*100
    out["gap_ru"]=out["urban"]-out["rural"]; out["gap_rp"]=out["W_richest"]-out["W_poorest"]
    preds_ru=["wealth","education",anc,"distance_big_problem"]
    c,g=decompose(a,"residence","urban",preds_ru); dm=_domains(c,DOM_RU); e=sum(dm.values())
    for k,v in dm.items(): out["RU_"+k]=v*100; out["RU_%_"+k]=v/g*100
    out["RU_explained_%"]=e/g*100; out["RU_unexplained_pp"]=(g-e)*100
    sub=a[a["wealth"].isin(["poorest","richest"])]; preds_rp=["residence","education",anc,"distance_big_problem"]
    c2,g2=decompose(sub,"wealth","richest",preds_rp); dm2=_domains(c2,DOM_RP); e2=sum(dm2.values())
    for k,v in dm2.items(): out["RP_"+k]=v*100; out["RP_%_"+k]=v/g2*100
    out["RP_explained_%"]=e2/g2*100; out["RP_unexplained_pp"]=(g2-e2)*100
    return out

def make_resampler(a):
    psu=a["v021"].astype(int).values; strat=a["v022"].astype(str).values
    s2c={s:np.unique(psu[strat==s]) for s in np.unique(strat)}
    crow={c:np.where(psu==c)[0] for c in np.unique(psu)}
    rng=np.random.default_rng(SEED)
    def draw():
        return np.concatenate([np.concatenate([crow[c] for c in rng.choice(cl,len(cl),replace=True)])
                               for cl in s2c.values()])
    return draw

def bootstrap_ci(a, statfun, n=N_BOOT):
    point=statfun(a); draw=make_resampler(a); keys=list(point.keys()); boot={k:[] for k in keys}
    for _ in range(n):
        st=statfun(a.iloc[draw()])
        for k in keys: boot[k].append(st[k])
    ci={k:(np.nanpercentile(boot[k],2.5), np.nanpercentile(boot[k],97.5)) for k in keys}
    return point, ci

## 7. Table 1 and Table 2 with 95% confidence intervals

In [ ]:
point, ci = bootstrap_ci(analysis, all_stats)

def show(label, key):
    print(f"  {label:22s} {point[key]:6.1f}  ({ci[key][0]:.1f}, {ci[key][1]:.1f})")

print("TABLE 1 — weighted facility-delivery prevalence (%, 95% CI)")
show("Overall","overall"); show("Urban","urban"); show("Rural","rural")
for q in ["poorest","poorer","middle","richer","richest"]: show("Wealth: "+q,"W_"+q)
for e in ["no education","primary","secondary or higher"]: show("Edu: "+e,"E_"+e)
show("ANC 4+","anc_yes"); show("ANC <4","anc_no")
show("Distance not a problem","dist_no"); show("Distance a problem","dist_yes")

print("\nTABLE 2 — decomposition (percentage points and % of gap, 95% CI)")
print("Rural-urban gap = %.1f (%.1f, %.1f)" % (point["gap_ru"], *ci["gap_ru"]))
for k in ["Wealth","Distance","Education","ANC 4+"]:
    print(f"  {k:16s} {point['RU_'+k]:5.1f} ({ci['RU_'+k][0]:.1f},{ci['RU_'+k][1]:.1f}) pp | "
          f"{point['RU_%_'+k]:5.1f} ({ci['RU_%_'+k][0]:.1f},{ci['RU_%_'+k][1]:.1f}) %")
print(f"  {'Explained %':16s} {point['RU_explained_%']:5.1f} ({ci['RU_explained_%'][0]:.1f},{ci['RU_explained_%'][1]:.1f})")
print(f"  {'Unexplained pp':16s} {point['RU_unexplained_pp']:5.1f} ({ci['RU_unexplained_pp'][0]:.1f},{ci['RU_unexplained_pp'][1]:.1f})")
print("Richest-poorest gap = %.1f (%.1f, %.1f)" % (point["gap_rp"], *ci["gap_rp"]))
for k in ["Urban residence","Education","Distance","ANC 4+"]:
    print(f"  {k:16s} {point['RP_'+k]:5.1f} ({ci['RP_'+k][0]:.1f},{ci['RP_'+k][1]:.1f}) pp | "
          f"{point['RP_%_'+k]:5.1f} ({ci['RP_%_'+k][0]:.1f},{ci['RP_%_'+k][1]:.1f}) %")
print(f"  {'Explained %':16s} {point['RP_explained_%']:5.1f} ({ci['RP_explained_%'][0]:.1f},{ci['RP_explained_%'][1]:.1f})")
print(f"  {'Unexplained pp':16s} {point['RP_unexplained_pp']:5.1f} ({ci['RP_unexplained_pp'][0]:.1f},{ci['RP_unexplained_pp'][1]:.1f})")

## 8. Linear-probability-model diagnostics
Proportion of fitted values outside [0, 1].

In [ ]:
def lpm_fit(d, group_col, hi, preds):
    d=d.dropna(subset=preds+["facility_delivery","weight"]).copy()
    X=pd.get_dummies(d[preds],drop_first=True).astype(float)
    y=d["facility_delivery"].astype(float).values; w=d["weight"].astype(float).values
    Xi=X.copy(); Xi.insert(0,"intercept",1.0); sw=np.sqrt(w)
    beta=np.linalg.lstsq(Xi.values*sw[:,None],y*sw,rcond=None)[0]
    return Xi.values@beta, len(d)
for name,d,gc,hi,preds in [("Rural-urban",analysis,"residence","urban",RU_PREDS),
    ("Richest-poorest",analysis[analysis["wealth"].isin(["poorest","richest"])],"wealth","richest",RP_PREDS)]:
    f,nn=lpm_fit(d,gc,hi,preds)
    print(f"{name}: n={nn}, fitted<0: {(f<0).sum()} ({(f<0).mean()*100:.1f}%), "
          f"fitted>1: {(f>1).sum()} ({(f>1).mean()*100:.1f}%), max={f.max():.2f}")

## 9. Wealth-by-residence interaction: formal test
Weighted linear probability model with an interaction; joint Wald test using the bootstrap covariance.

In [ ]:
d=analysis.dropna(subset=["facility_delivery","weight","wealth","residence"]).copy()
d["urban"]=(d["residence"]=="urban").astype(float)
wd=pd.get_dummies(d["wealth"],prefix="w",drop_first=True).astype(float)
inter=wd.mul(d["urban"].values,axis=0); inter.columns=["ux_"+c for c in wd.columns]
X=pd.concat([pd.Series(1.0,index=d.index,name="intercept"),d["urban"].rename("urban"),wd,inter],axis=1)
ixp=[list(X.columns).index(c) for c in inter.columns]
y=d["facility_delivery"].astype(float).values; w=d["weight"].astype(float).values
def wls(Xm,y,w): sw=np.sqrt(w); return np.linalg.lstsq(Xm*sw[:,None],y*sw,rcond=None)[0]
b=wls(X.values,y,w)[ixp]
psu=d["v021"].astype(int).values; strat=d["v022"].astype(str).values
s2c={s:np.unique(psu[strat==s]) for s in np.unique(strat)}; crow={c:np.where(psu==c)[0] for c in np.unique(psu)}
rng=np.random.default_rng(SEED); bs=[]
for _ in range(N_BOOT):
    idx=np.concatenate([np.concatenate([crow[c] for c in rng.choice(cl,len(cl),replace=True)]) for cl in s2c.values()])
    bs.append(wls(X.values[idx],y[idx],w[idx])[ixp])
Sig=np.cov(np.array(bs).T); Wald=b@np.linalg.pinv(Sig)@b; p=1-stats.chi2.cdf(Wald,len(ixp))
print(f"Joint Wald chi2({len(ixp)}) = {Wald:.2f}, p = {p:.3f}")

## 10. Sensitivity analyses
(a) ANC 8+ threshold instead of ANC 4+; (b) excluding ANC entirely.

In [ ]:
# (a) ANC 8+ threshold
p8,_=bootstrap_ci(analysis, lambda a: all_stats(a, anc="anc_8plus"), n=1)  # point only
print("ANC 8+ : rural-urban explained %.1f%% | richest-poorest explained %.1f%%"
      % (p8["RU_explained_%"], p8["RP_explained_%"]))
print("ANC 4+ : rural-urban explained %.1f%% | richest-poorest explained %.1f%%"
      % (point["RU_explained_%"], point["RP_explained_%"]))

# (b) excluding ANC
def dom_sum(a,gc,hi,preds,mapping):
    c,g=decompose(a,gc,hi,preds); dm=_domains(c,mapping); return dm,g,sum(dm.values())
for lab,gc,hi,mp,sub,pw,pn in [
    ("Rural-urban","residence","urban",DOM_RU,analysis,
        ["wealth","education","anc_4plus","distance_big_problem"],["wealth","education","distance_big_problem"]),
    ("Richest-poorest","wealth","richest",DOM_RP,analysis[analysis["wealth"].isin(["poorest","richest"])],
        ["residence","education","anc_4plus","distance_big_problem"],["residence","education","distance_big_problem"])]:
    for tag,preds in [("with ANC",pw),("without ANC",pn)]:
        dm,g,e=dom_sum(sub,gc,hi,preds,mp)
        line=", ".join(f"{k} {v*100:.1f}" for k,v in dm.items())
        print(f"{lab:16s} {tag:12s} explained {e/g*100:.1f}% | {line}")

## 11. Figures
Regional prevalence, and the wealth-by-residence interaction with 95% confidence intervals.

In [ ]:
# Regional lollipop
reg=(analysis.dropna(subset=["region","facility_delivery","weight"])
     .groupby("region",observed=True)
     .apply(lambda x: wmean(x["facility_delivery"].values,x["weight"].values)*100, include_groups=False)
     .sort_values())
plt.figure(figsize=(7,9))
plt.hlines(reg.index.str.title(), 0, reg.values, color="lightgray", lw=2)
plt.scatter(reg.values, reg.index.str.title(), color="#2E86AB", s=45)
plt.xlabel("Facility delivery (%)"); plt.xlim(0,105); plt.tight_layout(); plt.show()

In [ ]:
# Wealth x residence interaction with bootstrap 95% CIs
order=["poorest","poorer","middle","richer","richest"]
def cell_prev(a,q,r):
    s=a[(a["wealth"]==q)&(a["residence"]==r)]
    return wmean(s["facility_delivery"].values,s["weight"].values)*100 if len(s) else np.nan
draw=make_resampler(analysis)
bootc={(q,r):[] for q in order for r in ["urban","rural"]}
for _ in range(N_BOOT):
    samp=analysis.iloc[draw()]
    for q in order:
        for r in ["urban","rural"]: bootc[(q,r)].append(cell_prev(samp,q,r))
fig,ax=plt.subplots(figsize=(9,5.6)); col={"urban":"#2E86AB","rural":"#E1812C"}
for r in ["urban","rural"]:
    ys=[cell_prev(analysis,q,r) for q in order]
    lo=[ys[i]-np.nanpercentile(bootc[(q,r)],2.5) for i,q in enumerate(order)]
    hi=[np.nanpercentile(bootc[(q,r)],97.5)-ys[i] for i,q in enumerate(order)]
    ns=[int(((analysis["wealth"]==q)&(analysis["residence"]==r)).sum()) for q in order]
    ax.errorbar(range(5),ys,yerr=[lo,hi],marker="o",ms=7,lw=2.4,capsize=4,color=col[r],label=r.capitalize())
    for i,q in enumerate(order):
        ax.annotate(f"{ys[i]:.1f}%\n(n={ns[i]})",(i,ys[i]),textcoords="offset points",
                    xytext=(0,6 if r=="urban" else -9),ha="center",fontsize=8,color=col[r])
ax.set_xticks(range(5)); ax.set_xticklabels([q.title() for q in order])
ax.set_xlabel("Household wealth quintile"); ax.set_ylabel("Facility delivery (%)")
ax.set_ylim(30,105); ax.legend(title="Residence",loc="lower right"); ax.grid(axis="y",alpha=.25)
plt.tight_layout(); plt.show()